# General setup

In [87]:
# Imports
import matplotlib
matplotlib.use('Agg')
import argparse
import re
import json
import warnings
import numpy as np
from modules.CHILI import CHILI
from modules.net import SCVAE
from torch_geometric.loader import DataLoader
import torch
from torch.utils.data import TensorDataset
import datetime
import pathlib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ase import Atoms
from ase.io import write
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from modules.loss_functions import weighted_MSELoss, weighted_CrossEntropyLoss
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

In [88]:
# Functions
def create_cif(cell_params, cell_positions, cell_atoms, filename, prediction=True, composition=None, simplified_atom_identities=False):
    """
    Create a CIF file from the cell parameters, positions and atoms
    """
    if prediction:
        # Find argmax of atoms
        cell_atoms = np.argmax(cell_atoms, axis=1)

    # Remove atoms with atom number 0
    cell_positions = cell_positions[cell_atoms != 0]
    cell_atoms = cell_atoms[cell_atoms != 0]
    
    # Remove atoms not in the unit cell
    cell_atoms = cell_atoms[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    cell_positions = cell_positions[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    
    
    if simplified_atom_identities:
        cell_atoms = np.where(cell_atoms == 1, 8, cell_atoms)
        cell_atoms = np.where(cell_atoms == 2, 26, cell_atoms)
    
    # Create Atoms object
    atoms = Atoms(cell_atoms, scaled_positions=cell_positions, cell=cell_params)

    if not composition:
        composition = str(atoms.symbols)

    # Write CIF
    write(filename + f'.cif', images=atoms, format='cif') # _{composition}

    if not prediction:
        return composition
    return None

In [89]:
# Setup
model_path = './models/Supercell_beta_annealing_3d_latentMSE_biggerDecoder/' #'./models/Interpolation_Supercell_latentLog_beta_annealing_2d_latentMSE/'
model_name = 'best_model.pth' # 'best_model_annealed.pth'
setup_json_path = f'{model_path}setup_json.json'
with open(setup_json_path, 'r') as setup_json_file:
    setup_json = json.load(setup_json_file)

# Make prediction folders
experimental_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions'
pathlib.Path(experimental_folder).mkdir(parents=True, exist_ok=True)

interpolation_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions'
pathlib.Path(interpolation_folder).mkdir(parents=True, exist_ok=True)

sample_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions'
pathlib.Path(sample_folder).mkdir(parents=True, exist_ok=True)

# Make paper figures folder
paper_figures_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures'
pathlib.Path(paper_figures_folder).mkdir(parents=True, exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [90]:
# Load model
model = SCVAE(
    latent_dim=setup_json['model']['latent_dim'],
    out_dim=setup_json['model']['out_dim'],
    prior_factor=setup_json['model']['prior_factor'],
    gnn_dim=setup_json['model']['gnn_dim'],
    gnn_heads=setup_json['model']['gnn_heads'],
    gnn_edge_dim=setup_json['model']['gnn_edge_dim'],
    scattering_channels=setup_json['model']['scattering_channels'],
    scattering_dim=setup_json['model']['scattering_dim'],
    scattering_kernel_size=setup_json['model']['scattering_kernel_size'],
    scattering_stride=setup_json['model']['scattering_stride'],
    scattering_padding=setup_json['model']['scattering_padding'],
    composition_dim=setup_json['model']['composition_dim'],
    decoder_hidden_dim=setup_json['model']['decoder_hidden_dim'],
    position_output_dim=setup_json['model']['position_output_dim'],
    atom_output_dim=setup_json['model']['atom_output_dim'],
    cell_output_dim=setup_json['model']['cell_output_dim'],
    seperate_decoder=setup_json['training']['seperate_decoder'],
).to(device)

# Load model weights
model.load_state_dict(torch.load(model_path + model_name))

<All keys matched successfully>

In [91]:
# Load normalization parameters
if setup_json['data']['normalize_cell_parameters']:
    cell_means = torch.tensor([
        setup_json['data']['cell_normalization']['a']['mean'],
        setup_json['data']['cell_normalization']['b']['mean'],
        setup_json['data']['cell_normalization']['c']['mean'],
        setup_json['data']['cell_normalization']['alpha']['mean'],
        setup_json['data']['cell_normalization']['beta']['mean'],
        setup_json['data']['cell_normalization']['gamma']['mean'],
    ]).float().to(device)
    cell_stds = torch.tensor([
        setup_json['data']['cell_normalization']['a']['std'],
        setup_json['data']['cell_normalization']['b']['std'],
        setup_json['data']['cell_normalization']['c']['std'],
        setup_json['data']['cell_normalization']['alpha']['std'],
        setup_json['data']['cell_normalization']['beta']['std'],
        setup_json['data']['cell_normalization']['gamma']['std'],
    ]).float().to(device)

if setup_json['data']['normalize_atom_positions']:
    atom_position_means = torch.tensor(setup_json['data']['atom_position_normalization']['mean']).float().to(device)
    atom_position_stds = torch.tensor(setup_json['data']['atom_position_normalization']['std']).float().to(device)

if setup_json['data']['normalize_distances']:
    distance_means = torch.tensor(setup_json['data']['distance_normalization']['mean']).float().to(device)
    distance_stds = torch.tensor(setup_json['data']['distance_normalization']['std']).float().to(device)

beta = setup_json['training']['beta']
out_dim = setup_json['model']['out_dim']

# Load model test results

## Code

In [92]:
# Load results from test script
with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/losses_{model_name[:-4]}.json', 'r') as f:
    losses_json = json.load(f)
df_losses = pd.DataFrame(losses_json)

with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/reconstructions_{model_name[:-4]}.json', 'r') as f:
    rec_json = json.load(f)
df_rec = pd.DataFrame(rec_json)
# Capitalize crystal types if not already
name_map = {'interpolated': 'Interpolated', 'rocksalt': 'RockSalt', 'spinel': 'Spinel', 'zincblende': 'ZincBlende'}
df_rec['crystalType'] = df_rec['crystalType'].apply(lambda x: name_map[x] if x in name_map else x)

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    df_rec[['ls1', 'ls2']] = df_rec['latent_space_mean'].apply(pd.Series)
else:
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1', 'pc2']] = pca.transform(np.array(df_rec['latent_space_mean'].values.tolist()))

In [93]:
df_losses.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,crystalType,particleSize
0,-6.282568,0.000547,0.000200,0.000335,0.000012,0.001305,RheniumTrioxide,11.694150
1,-3.569565,0.010295,0.006306,0.003942,0.000048,0.017426,Wurtzite,23.968201
2,-2.222544,0.002417,0.002201,0.000213,0.000003,0.104146,CaesiumChloride,14.847806
3,-5.926445,0.000246,0.000168,0.000064,0.000015,0.002386,RheniumTrioxide,53.604038
4,-7.649301,0.000187,0.000128,0.000051,0.000008,0.000285,RheniumTrioxide,11.700600


In [94]:
df_rec.head()

,crystalType,n_atoms,n_oxygens,n_metals,cell_parameters,cell_positions,cell_atoms,latent_space_mean,latent_space_std,true_cell_parameters,true_cell_positions,true_cell_atoms,pc1,pc2
0,RheniumTrioxide,56,36,20,"[3.878023386001587, 3.9418554306030273, 3.9107...","[[-0.0024300001095980406, -0.00566999986767768...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.23553481698036194, 0.14010709524154663, 0.8...","[0.039461951702833176, 0.3383505940437317, 0.0...","[-0.3189188539981842, -0.3189188539981842, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[32, 8, 8, 8, 32, 32, 32, 8, 8, 8, 8, 8, 8, 32...",-0.708994,0.164626
1,Wurtzite,56,25,31,"[3.859593391418457, 3.8858187198638916, 6.2735...","[[0.3006500005722046, 0.6079000234603882, -0.0...","[2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 2, 1, 1, 2, 2, ...","[0.22790616750717163, 0.21206140518188477, 0.0...","[0.05273773521184921, 0.3450313210487366, 0.06...","[-0.4546557068824768, -0.4546557068824768, 0.0...","[[0.3333300054073334, 0.666670024394989, 0.0],...","[11, 8, 11, 8, 11, 11, 8, 8, 11, 11, 11, 8, 8,...",-0.012824,-0.225236
2,CaesiumChloride,56,20,36,"[3.279444694519043, 3.2935566902160645, 3.2519...","[[0.0017000000225380063, -0.003370000049471855...","[2, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, ...","[0.7565188407897949, 0.2613319158554077, -0.43...","[0.049608174711465836, 0.35037463903427124, 0....","[-0.6175447106361389, -0.6175447106361389, -0....","[[0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.0, 0.0, ...","[19, 8, 19, 19, 19, 19, 19, 19, 8, 8, 8, 19, 1...",0.130011,-0.916179
3,RheniumTrioxide,56,36,20,"[3.87994647026062, 3.9403750896453857, 3.93291...","[[0.007400000002235174, 0.0019600000232458115,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.20122650265693665, 0.13083508610725403, 0.8...","[0.04307560622692108, 0.338651567697525, 0.052...","[-0.3182690441608429, -0.3182690441608429, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[73, 8, 8, 8, 73, 73, 73, 8, 8, 8, 8, 8, 8, 73...",-0.697878,0.197799
4,RheniumTrioxide,56,36,20,"[3.863217830657959, 3.9153716564178467, 3.8750...","[[-0.0004199999966658652, 0.009519999846816063...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.205844908952713, 0.1337863802909851, 0.8331...","[0.0393567830324173, 0.3347398042678833, 0.048...","[-0.31761327385902405, -0.31761327385902405, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[72, 8, 8, 8, 72, 72, 72, 8, 8, 8, 8, 8, 8, 72...",-0.703554,0.195726


In [95]:
df_combined = pd.concat([df_losses.drop('crystalType', axis=1), df_rec], axis=1)
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,cell_parameters,cell_positions,cell_atoms,latent_space_mean,latent_space_std,true_cell_parameters,true_cell_positions,true_cell_atoms,pc1,pc2
0,-6.282568,0.000547,0.000200,0.000335,0.000012,0.001305,11.694150,RheniumTrioxide,56,36,...,"[3.878023386001587, 3.9418554306030273, 3.9107...","[[-0.0024300001095980406, -0.00566999986767768...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.23553481698036194, 0.14010709524154663, 0.8...","[0.039461951702833176, 0.3383505940437317, 0.0...","[-0.3189188539981842, -0.3189188539981842, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[32, 8, 8, 8, 32, 32, 32, 8, 8, 8, 8, 8, 8, 32...",-0.708994,0.164626
1,-3.569565,0.010295,0.006306,0.003942,0.000048,0.017426,23.968201,Wurtzite,56,25,...,"[3.859593391418457, 3.8858187198638916, 6.2735...","[[0.3006500005722046, 0.6079000234603882, -0.0...","[2, 1, 2, 1, 2, 2, 1, 1, 2, 2, 2, 1, 1, 2, 2, ...","[0.22790616750717163, 0.21206140518188477, 0.0...","[0.05273773521184921, 0.3450313210487366, 0.06...","[-0.4546557068824768, -0.4546557068824768, 0.0...","[[0.3333300054073334, 0.666670024394989, 0.0],...","[11, 8, 11, 8, 11, 11, 8, 8, 11, 11, 11, 8, 8,...",-0.012824,-0.225236
2,-2.222544,0.002417,0.002201,0.000213,0.000003,0.104146,14.847806,CaesiumChloride,56,20,...,"[3.279444694519043, 3.2935566902160645, 3.2519...","[[0.0017000000225380063, -0.003370000049471855...","[2, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, ...","[0.7565188407897949, 0.2613319158554077, -0.43...","[0.049608174711465836, 0.35037463903427124, 0....","[-0.6175447106361389, -0.6175447106361389, -0....","[[0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.0, 0.0, ...","[19, 8, 19, 19, 19, 19, 19, 19, 8, 8, 8, 19, 1...",0.130011,-0.916179
3,-5.926445,0.000246,0.000168,0.000064,0.000015,0.002386,53.604038,RheniumTrioxide,56,36,...,"[3.87994647026062, 3.9403750896453857, 3.93291...","[[0.007400000002235174, 0.0019600000232458115,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.20122650265693665, 0.13083508610725403, 0.8...","[0.04307560622692108, 0.338651567697525, 0.052...","[-0.3182690441608429, -0.3182690441608429, -0....","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[73, 8, 8, 8, 73, 73, 73, 8, 8, 8, 8, 8, 8, 73...",-0.697878,0.197799
4,-7.649301,0.000187,0.000128,0.000051,0.000008,0.000285,11.700600,RheniumTrioxide,56,36,...,"[3.863217830657959, 3.9153716564178467, 3.8750...","[[-0.0004199999966658652, 0.009519999846816063...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...","[0.205844908952713, 0.1337863802909851, 0.8331...","[0.0393567830324173, 0.3347398042678833, 0.048...","[-0.31761327385902405, -0.31761327385902405, -...","[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 0.0, ...","[72, 8, 8, 8, 72, 72, 72, 8, 8, 8, 8, 8, 8, 72...",-0.703554,0.195726


In [96]:
# Load latent log if it exists
import yaml
try:
    df_latent_log = pd.read_csv(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_log.csv')

    df_latent_log = df_latent_log.drop(df_latent_log[df_latent_log['posterior_mean'] == 'posterior_mean'].index)

    # df_latent_log = df_latent_log[:1000]

    # Select only every tenth epoch
    # Convert epoch to ints
    df_latent_log['epoch'] = df_latent_log['epoch'].astype(np.int16)

    df_latent_log = df_latent_log[df_latent_log['epoch'] % 10 == 0]

    df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']] = df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']].applymap(yaml.safe_load)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_latent_log[['ls1', 'ls2']] = df_latent_log['posterior_mean'].apply(pd.Series)
        df_latent_log[['ls1_prior', 'ls2_prior']] = df_latent_log['prior_mean'].apply(pd.Series)
    else:
        # Perform PCA
        # pca = PCA(n_components=2)
        # pca.fit(np.array(df_latent_log['posterior_mean'].values.tolist()))
        df_latent_log[['pc1', 'pc2']] = pca.transform(np.array(df_latent_log['posterior_mean'].values.tolist()))

        df_latent_log[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_latent_log['prior_mean'].values.tolist()))
        
    print(df_latent_log.columns)
except:
    df_latent_log = None
    print('No latent log found')

No latent log found


## Plot

In [97]:
# Set seaborn style
# sns.set_theme(style='darkgrid', font_scale=1.)
# Make animation of latent space throughout 
# Plot latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_means.pdf', dpi=300)

In [98]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_combined[contour_column].min()
max_loss = df_combined[contour_column].max()

# Plot
if len(df_combined['latent_space_mean'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['ls1'].min()*1.1, df_combined['ls1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['ls2'].min()*1.1, df_combined['ls2'].max()*1.1, 1000)
    zi = griddata((df_combined['ls1'], df_combined['ls2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['pc1'].min()*1.1, df_combined['pc1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['pc2'].min()*1.1, df_combined['pc2'].max()*1.1, 1000)
    zi = griddata((df_combined['pc1'], df_combined['pc2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


In [99]:
# # 2D scatter plot with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         pass
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace)
        
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types):
#             fig.data[i]['visible'] = True
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j, crystal_type in enumerate(crystal_types):
#                 step['args'][1][i*n_crystal_types + j] = True
#             steps.append(step)    
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.show()

In [100]:
# Sample n points from each distribution in the latent space log
if df_latent_log is not None:
    n_samples = 100
    
    crystal_type_list = []
    latent_space_list = []
    epoch_list = []
    for latent_mean, latent_std, crystal_type, epoch in zip(df_latent_log['posterior_mean'], df_latent_log['posterior_std'], df_latent_log['crystal_type'], df_latent_log['epoch']):
        latent_space_samples = torch.distributions.Normal(loc=torch.tensor(latent_mean), scale=torch.tensor(latent_std)).sample((n_samples,)).numpy()
        latent_space_list.extend(latent_space_samples)
        crystal_type_list.extend([crystal_type] * n_samples)
        epoch_list.extend([epoch] * n_samples)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_log_samples = pd.DataFrame(latent_space_list, columns=['ls1', 'ls2'])
    else:
        latent_space_list = pca.transform(latent_space_list)
        df_log_samples = pd.DataFrame(latent_space_list, columns=['pc1', 'pc2'])

    df_log_samples['crystalType'] = crystal_type_list
    df_log_samples['epoch'] = epoch_list

In [101]:
# # 2D scatter plot and 2D histogram with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['ls1'],
#                     y=df_crystal['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['ls1'],
#                     y=df_crystal_samples['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
            
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)
            
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='LS dim 1')
#         fig.update_yaxes(title_text='LS dim 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['pc1'],
#                     y=df_crystal_samples['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
        
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)  
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
    

In [102]:
# Plot the latent space at specific epochs
epoch_to_plot = 0

if df_latent_log is not None:
    plt.figure(figsize=(10, 8))
    df_epoch = df_latent_log[df_latent_log['epoch'] == epoch_to_plot]
    df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch_to_plot]
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        sns.scatterplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['ls1'].min(), df_log_samples['ls1'].max())
        plt.ylim(df_log_samples['ls2'].min(), df_log_samples['ls2'].max())  
    else:
        sns.scatterplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['pc1'].min(), df_log_samples['pc1'].max())
        plt.ylim(df_log_samples['pc2'].min(), df_log_samples['pc2'].max())
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # plt.axis('equal')
    plt.tight_layout()
    plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epoch_{epoch_to_plot}.pdf', dpi=300)

In [103]:
# Make a combined figure of the latent space at different epochs
epochs_to_plot = [0, 10, 100, 350, 690, 890]
subplot_rows = 2
subplot_cols = 3
figsize = (10, 6.5)

if df_latent_log is not None:
    fig, ax = plt.subplots(subplot_rows, subplot_cols, figsize=figsize, sharex=True, sharey=True)
    ax = ax.flatten()
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        xlim_min = df_log_samples['ls1'].min()
        xlim_max = df_log_samples['ls1'].max()
        ylim_min = df_log_samples['ls2'].min()
        ylim_max = df_log_samples['ls2'].max()
    else:
        xlim_min = df_log_samples['pc1'].min()
        xlim_max = df_log_samples['pc1'].max()
        ylim_min = df_log_samples['pc2'].min()
        ylim_max = df_log_samples['pc2'].max()
    
    for i, epoch in enumerate(epochs_to_plot):
        df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
        df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
        
        if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
            sns.histplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        else:
            sns.histplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('PC 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('PC 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        if i == 0:
            handles, labels = ax[i].get_legend_handles_labels()
            ax[i].legend(loc='lower center', bbox_to_anchor=(subplot_cols/2, 1), ncol=4)
        else:
            ax[i].legend([],[], frameon=False)
        ax[i].set_title(f' Epoch {epoch}', fontsize=12, fontweight='bold', y=1.0, pad=-15, loc='left')
    
    # Remove empty axes
    for i in range(len(epochs_to_plot), subplot_rows*subplot_cols):
        fig.delaxes(ax[i])
        
    # fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 3), ncols=5)
    
    plt.tight_layout()
    
    # Remove whitespace
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    
    plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epochs.pdf', dpi=300)

# Experimental data

## Code

In [104]:
data_folder = './data/Experimental/'

# Load experimental data
data_paths = [str(p) for p in pathlib.Path(data_folder).rglob('*.gr')]

data_filepath = []
data_composition_string = []
data_composition = []
data_pdf = []
for data_path in data_paths:
    with open(data_path, 'r') as f:
        # Load data
        line_counter = 0
        for line in f:
            if line.startswith('composition'):
                composition = ''.join(line.split(' ')[2:])
            if line.startswith('0'):
                header_line = line_counter
                break
            line_counter += 1
    # Remove line breaks
    composition = composition.replace('\n', '')
    
    # Save composition string
    composition_string = deepcopy(composition)

    # Remove stochiometry from composition
    composition = re.sub(r'[0-9\.]+', '', composition)

    # # Split string on capital letters
    composition = re.findall('[A-Z][^A-Z]*', composition)

    # Translate composition to atom numbers
    composition = Atoms(symbols=composition).get_atomic_numbers()
    
    composition_onehot = np.zeros(119)
    composition_onehot[composition] = 1
    
    
    # Load data
    data = pd.read_csv(data_path, sep=' ', skiprows=header_line, names=['r [Å]', 'G(r) [Å⁻²]'])
    
    data_r = np.arange(0,60,0.01)
    data_Gr = np.interp(data_r, data['r [Å]'], data['G(r) [Å⁻²]'], left=0, right=0)
    data_Gr = data_Gr / np.amax(data_Gr)
    
    data_filepath.append(data_path)
    data_composition_string.append(composition_string)
    data_composition.append(composition_onehot)
    data_pdf.append(data_Gr)

# Convert to tensors
data_composition = torch.tensor(data_composition, dtype=torch.long)
data_pdf = torch.tensor(data_pdf, dtype=torch.float32)
data_composition_string_index = torch.tensor(np.arange(len(data_composition_string)))
data_filepath_index = torch.tensor(np.arange(len(data_filepath)))

exp_data = TensorDataset(data_pdf, data_composition, data_composition_string_index, data_filepath_index)

# Dataloader
exp_loader = DataLoader(exp_data, batch_size=10, shuffle=False)

# Results dict
exp_results = {
    'composition': [],
    'pdf': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [105]:
# Inference
model.eval()
for batch in tqdm(exp_loader, desc='Inference', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    pdf, composition, composition_string_index, filepath_index = batch
    pdf = pdf.unsqueeze(-1).to(device)
    composition = composition.float().to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store composition
    for index in composition_string_index:
        exp_results['composition'].append(data_composition_string[index])
    
    # Store PDF
    exp_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store latent representation
    exp_results['prior_mean'].extend(prior_mean.cpu().tolist())
    exp_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    exp_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    exp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    exp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    exp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
                prediction=True,
                composition=data_composition_string[composition_string_index[batch_index]],
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_exp_results = pd.DataFrame(exp_results)
if len(df_exp_results['prior_mean'].iloc[0]) == 2:
    df_exp_results[['ls1', 'ls2']] = df_exp_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_exp_results[['pc1', 'pc2']] = pca.transform(np.array(df_exp_results['prior_mean'].values.tolist()))
df_exp_results.head()

# Drop IrO2
# df_exp_results = df_exp_results[df_exp_results['composition'] != 'IrO2']

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,pc1,pc2
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[0.30368298292160034, 0.27777403593063354, -0....","[0.06663680076599121, 0.3582204282283783, 0.08...","[0.31661176681518555, 1.0520222187042236, -0.8...","[4.586600303649902, 4.623684883117676, 2.93155...","[[0.001970000099390745, 0.006149999797344208, ...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.711705,-0.727443
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-1.3286439180374146, 0.2882087230682373, -0.4...","[0.07203423976898193, 0.3682563304901123, 0.07...","[-1.2674825191497803, -0.4204633831977844, -0....","[8.317983627319336, 8.346033096313477, 8.28006...","[[0.4915499985218048, 0.4876300096511841, 0.49...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",1.142011,0.907139
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[0.32437655329704285, 0.28595924377441406, -0....","[0.06868987530469894, 0.35595473647117615, 0.0...","[0.2797665596008301, -0.005042552947998047, -0...","[4.591385364532471, 4.64390230178833, 2.926997...","[[0.0039900001138448715, -0.003460000036284327...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...",0.759420,-0.778614
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[-0.46725237369537354, 0.24489690363407135, -0...","[0.09540387243032455, 0.39881643652915955, 0.0...","[-0.5889257192611694, -0.4179863929748535, -0....","[5.48960542678833, 5.54686975479126, 5.5289082...","[[0.006829999852925539, -0.006130000110715628,...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",1.068403,-0.043300
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.4730594754219055, 0.24492371082305908, -0....","[0.09430854022502899, 0.3988487422466278, 0.09...","[-0.48178961873054504, 0.32438918948173523, -0...","[5.325958251953125, 5.3862104415893555, 5.3704...","[[0.005189999938011169, -0.000910000002477318,...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...",1.077978,-0.042091


In [106]:
df_exp_results['ground_truth_crystal_type'] = None
df_exp_results['ground_truth_crystal_type'].iloc[0:3] = 'Rutile'
df_exp_results['ground_truth_crystal_type'].iloc[3:6] = 'Fluorite'
df_exp_results['ground_truth_crystal_type'].iloc[6:11] = 'Spinel'

## Plot

In [107]:
# Plot latent space placement of experimental data
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='ls1', y='ls2', hue='composition', style='ground_truth_crystal_type', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='pc1', y='pc2', c='black', style='composition', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_samples.pdf', dpi=300)

In [108]:
# Plot the experimental data
# index_list = [0,1,2] # Rutile samples
index_list = [3, 4, 5] # Fluorite samples
# index_list = list(range(6,11)) # Spinel samples

plt.figure(figsize=(8, 6))
for index in index_list:
    plt.plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
plt.yticks([])
# plt.legend(ncols=2, loc='lower center', bbox_to_anchor=(0.5, 1.01))
plt.legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# plt.title(f'{df_exp_results["composition"].iloc[index]}', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Rutile.pdf', dpi=300)
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Fluorite.pdf', dpi=300)
# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Spinel.pdf', dpi=300)

# Interpolate in latent space

## Code

In [109]:
# Create latent samples for interpolation between two points
n_latent_samples = 10

# Define two latent points from pca space
# latent_point_1 = (24, 1) # ex1 (9, -13)
# latent_point_2 = (24, -15) # ex1 (24,-9)

# Define n latent points from latent space
crystal_types = ['RockSalt', 'Spinel', 'ZincBlende'] #['CadmiumIodide', 'CadmiumChloride'] #['RockSalt', 'Spinel', 'ZincBlende']
indices = [0, 2, 4] #[14,3] #[0, 4, 0]
point_indices = [df_rec[(df_rec['crystalType'] == crystal_type)].index[index] for crystal_type, index in zip(crystal_types, indices)]

latent_points = [np.array(df_rec['latent_space_mean'].iloc[point_index]) for point_index in point_indices]

# Interpolation mode flag (True for PCA. False for full latent space)
pca_inter = False

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Latent samples are n_latent_samples evenly spaced points between the two points
if pca_inter:
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_pca = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_pca = np.concatenate([interp_pca, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T], axis=0)
    
    if len(df_rec['latent_space_mean'].iloc[0]) > 2:
        interp_latent = pca.inverse_transform(interp_pca)
    else:
        interp_latent = interp_pca
    interp_latent = torch.tensor(interp_latent).float()
else:
    # Transform back to latent dimensions if latent space is more than 2D
    if (len(df_rec['latent_space_mean'].iloc[0]) > 2) and (len(latent_points[0]) == 2):
        latent_points = pca.inverse_transform(latent_points)
    
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_latent = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_latent = np.concatenate([interp_latent, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T], axis=0)
        
    interp_latent = torch.tensor(interp_latent).float()

n_interp_points = interp_latent.shape[0]
interp_index = torch.tensor([i for i in range(n_interp_points)])

# Create dataset
interp_dataset = TensorDataset(interp_latent, interp_index, comp_onehot.repeat(n_interp_points, 1))

# Dataloader
interp_loader = DataLoader(interp_dataset, batch_size=10, shuffle=False)

# Results dict
interp_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [110]:
# Inference
model.eval()
for batch in tqdm(interp_loader, desc='Interpolating', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    interp_point, interp_index, composition = batch
    interp_point = interp_point.to(device)
    interp_index = interp_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            interp_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    interp_results['name'].extend([f'sample_{interp_index[batch_index]}' for batch_index in range(this_batch_size)])
    interp_results['latent_point'].extend(interp_point.cpu().tolist())
    
    # Store predictions
    interp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions/sample{interp_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {interp_index[batch_index]}.')
    

In [111]:
df_interp = pd.DataFrame(interp_results)
if len(df_interp['latent_point'].iloc[0]) == 2:
    df_interp[['ls1', 'ls2']] = df_interp['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_interp[['pc1', 'pc2']] = pca.transform(np.array(df_interp['latent_point'].values.tolist()))
df_interp.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,pc1,pc2
0,sample_0,"[0.02070300281047821, 0.2536712884902954, -0.4...","[4.275302886962891, 4.2940754890441895, 4.2612...","[[-0.004579999949783087, -0.012959999963641167...","[2, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, ...",0.514794,-0.288597
1,sample_1,"[-0.13574132323265076, 0.25777798891067505, -0...","[4.723681926727295, 4.744466781616211, 4.70446...","[[-0.0013000000035390258, -0.00571000017225742...","[2, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, ...",0.584535,-0.148257
2,sample_2,"[-0.2921856641769409, 0.2618847191333771, -0.4...","[5.447658538818359, 5.46982479095459, 5.442946...","[[-0.0016700000269338489, -0.00643000006675720...","[2, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, ...",0.654276,-0.007917
3,sample_3,"[-0.4486299753189087, 0.2659914195537567, -0.4...","[5.915637016296387, 5.979872703552246, 5.90371...","[[-0.014039999805390835, -0.023339999839663506...","[2, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, ...",0.724018,0.132423
4,sample_4,"[-0.6050742864608765, 0.27009814977645874, -0....","[6.244915008544922, 6.242930889129639, 6.17778...","[[0.032269999384880066, -0.014100000262260437,...","[2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 1, ...",0.793759,0.272763


## Plot

In [112]:
# Plot interpolation results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_samples.pdf', dpi=300)

In [113]:
# Plot the cell parameters of the interpolation with angles and lengths on two subplots (one for lengths and one for angles) that share the x-axis
df_cell_params = pd.DataFrame(df_interp['cell_parameters'].tolist(), columns=['a', 'b', 'c', 'alpha', 'beta', 'gamma'])
df_cell_params['name'] = df_interp['name']

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Lengths
sns.lineplot(data=df_cell_params[['a', 'b', 'c']], ax=ax0)
ax0.set_ylabel('Length [Å]', fontsize=14, fontweight='bold')

# Angles
sns.lineplot(data=df_cell_params[['alpha', 'beta', 'gamma']], ax=ax1)
ax1.set_ylabel('Angle [°]', fontsize=14, fontweight='bold')
ax1.set_xlabel('Interpolation sample #', fontsize=14, fontweight='bold')
# y-axis label and ticks on the right side
ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")

# Add vertical lines for the interpolation outside clusters and color between them
line_1 = 1.5
line_2 = 6.5
line_3 = 12.5
line_4 = 17.5
ax0.axvline(x=line_1, color='red', linestyle='--')
ax0.axvline(x=line_2, color='red', linestyle='--')
ax0.axvspan(line_1, line_2, color='red', alpha=0.1)

ax0.axvline(x=line_3, color='red', linestyle='--')
ax0.axvline(x=line_4, color='red', linestyle='--')
ax0.axvspan(line_3, line_4, color='red', alpha=0.1)

ax1.axvline(x=line_1, color='red', linestyle='--')
ax1.axvline(x=line_2, color='red', linestyle='--')
ax1.axvspan(line_1, line_2, color='red', alpha=0.1)

ax1.axvline(x=line_3, color='red', linestyle='--')
ax1.axvline(x=line_4, color='red', linestyle='--')
ax1.axvspan(line_3, line_4, color='red', alpha=0.1)

plt.xlim(0, n_interp_points-1)

# Legends
ax0.legend(loc='lower center' , bbox_to_anchor=(0.5, 0.05), ncol=3)
ax1.legend(loc='upper center' , bbox_to_anchor=(0.5, 0.95), ncol=3)

# Make x-ticks integers
plt.xticks(np.arange(0, n_interp_points, 1))

fig.tight_layout()

# Remove space between subplots
plt.subplots_adjust(hspace=0)

plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/interpolation_cell_parameters.pdf', dpi=300)

# Sample same distribution multiple times

## Code

In [135]:
# Find index of rows containing 26 in the true_cell_atoms column
atomic_number_of_interest = 29
index_list = []
for i, row in df_rec.iterrows():
    if atomic_number_of_interest in row['true_cell_atoms']:
        index_list.append(i)
index_list

[26, 133, 170, 250, 255, 317]

In [136]:
df_rec.iloc[index_list]

,crystalType,n_atoms,n_oxygens,n_metals,cell_parameters,cell_positions,cell_atoms,latent_space_mean,latent_space_std,true_cell_parameters,true_cell_positions,true_cell_atoms,pc1,pc2
26,Fluorite,56,30,26,"[5.290012359619141, 5.351812839508057, 5.33472...","[[0.0051299999468028545, 0.0005200000014156103...","[2, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, ...","[-0.39844581484794617, 0.2686007022857666, -0....","[0.06930136680603027, 0.37610265612602234, 0.0...","[0.48128533363342285, 0.48128533363342285, -0....","[[0.0, 0.0, 0.0], [0.25, 0.25, 0.25], [0.0, 0....","[29, 8, 29, 29, 29, 8, 8, 8, 29, 29, 29, 8, 8,...",1.183134,-0.188292
133,Spinel,56,32,24,"[7.947784423828125, 7.984474182128906, 7.89815...","[[0.48153001070022583, 0.4802300035953522, 0.4...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[-1.2126282453536987, 0.290932297706604, -0.40...","[0.0504937581717968, 0.3532828092575073, 0.057...","[2.269524097442627, 2.269524097442627, 0.75015...","[[0.5, 0.5, 0.5], [0.5, 0.0, 0.0], [0.0, 0.0, ...","[29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 2...",1.079822,0.809058
170,AntiFluorite,56,26,30,"[4.992137908935547, 4.9824538230896, 4.9403662...","[[-0.01665000058710575, -0.003220000071451068,...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...","[-0.32517939805984497, 0.19376853108406067, 0....","[0.047208286821842194, 0.32444077730178833, 0....","[0.2970680594444275, 0.2970680594444275, -0.25...","[[0.0, 0.0, 0.0], [0.25, 0.25, 0.25], [0.5, 0....","[8, 29, 8, 8, 8, 29, 29, 29, 8, 8, 8, 29, 29, ...",-0.203764,0.521438
250,AntiFluorite,56,26,30,"[4.902187824249268, 4.9024577140808105, 4.8277...","[[-0.011009999550879002, -0.000419999996665865...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...","[-0.3194969594478607, 0.1901392936706543, 0.55...","[0.045799609273672104, 0.32410940527915955, 0....","[0.2970680594444275, 0.2970680594444275, -0.25...","[[0.0, 0.0, 0.0], [0.25, 0.25, 0.25], [0.5, 0....","[8, 29, 8, 8, 8, 29, 29, 29, 8, 8, 8, 29, 29, ...",-0.202530,0.514186
255,CadmiumIodide,56,30,26,"[2.7876901626586914, 2.899672031402588, 4.4229...","[[2.9999999242136255e-05, 0.014619999565184116...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...","[0.7879116535186768, 0.1653820276260376, 1.163...","[0.06949298083782196, 0.38388538360595703, 0.0...","[-0.9476407170295715, -0.9476407170295715, -0....","[[0.0, 0.0, 0.0], [0.3333300054073334, 0.66667...","[29, 8, 29, 29, 29, 8, 29, 29, 29, 8, 8, 8, 8,...",-1.277453,-0.146411
317,ZincBlende,56,20,36,"[4.274829864501953, 4.278858184814453, 4.32420...","[[-0.01128000020980835, 0.0026700000744313, -0...","[2, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...","[-0.22437119483947754, 0.2175232172012329, -0....","[0.03626646101474762, 0.3305405378341675, 0.04...","[-0.019801003858447075, -0.019801003858447075,...","[[0.0, 0.0, 0.0], [0.25, 0.25, 0.25], [0.0, 0....","[29, 8, 29, 29, 29, 29, 29, 29, 8, 8, 8, 29, 2...",0.277668,0.129601


In [139]:
# Number of samples to generate
n_samples = 10

# Latent distribution to sample from
dist_crystal_type = 'AntiFluorite'
crystal_type_index = 0
dist_index = df_rec[(df_rec['crystalType'] == dist_crystal_type)].index[crystal_type_index]
dist_mean = df_rec.latent_space_mean.iloc[dist_index]
dist_std = df_rec.latent_space_std.iloc[dist_index]

# Latent distribution from experimental data
# exp_index = 3
# dist_mean = df_exp_results.prior_mean.iloc[exp_index]
# dist_std = df_exp_results.prior_log_std.iloc[exp_index]

latent_dist = torch.distributions.Normal(loc=torch.tensor(dist_mean).float(), scale=torch.tensor(dist_std).float())

# Sample from latent distribution
latent_mean = latent_dist.mean
latent_samples = latent_dist.sample((n_samples,))
# Concatenate with latent mean
latent_samples = torch.cat((latent_mean.unsqueeze(0), latent_samples), dim=0)
sample_index = torch.tensor([i for i in range(n_samples+1)])

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Define dataset
sample_dataset = TensorDataset(latent_samples, sample_index, comp_onehot.repeat(n_samples+1, 1))

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [140]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition = batch
    sample_point = sample_point.to(device)
    sample_index = sample_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    sample_results['name'].extend([f'sample_{sample_index[batch_index]}' for batch_index in range(this_batch_size)])
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions/sample{sample_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {sample_index[batch_index]}.')

In [141]:
df_sample = pd.DataFrame(sample_results)
if len(df_sample['latent_point'].iloc[0]) == 2:
    df_sample[['ls1', 'ls2']] = df_sample['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_sample[['pc1', 'pc2']] = pca.transform(np.array(df_sample['latent_point'].values.tolist()))
    
df_sample['name'][df_sample['name'] == 'sample_0'] = 'Dist. mean'
df_sample.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,pc1,pc2
0,Dist. mean,"[-0.3050370514392853, 0.1859518587589264, 0.56...","[4.859355926513672, 4.852563381195068, 4.77281...","[[-0.010320000350475311, -0.001599999959580600...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...",-0.212396,0.503174
1,sample_1,"[-0.2581562101840973, 0.27743208408355713, 0.5...","[4.743903160095215, 4.736494064331055, 4.65141...","[[-0.012109999544918537, 0.0001500000071246177...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...",-0.201008,0.442590
2,sample_2,"[-0.3001387119293213, -0.04040458798408508, 0....","[4.83966064453125, 4.837599754333496, 4.758657...","[[-0.011020000092685223, 0.0010999999940395355...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...",-0.198257,0.489535
3,sample_3,"[-0.23424309492111206, 0.4091201722621918, 0.5...","[4.701971054077148, 4.679181098937988, 4.58441...","[[-0.011140000075101852, -0.0088900001719594, ...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...",-0.265541,0.451927
4,sample_4,"[-0.3362899124622345, 0.2087680697441101, 0.39...","[4.887899875640869, 4.897553443908691, 4.86029...","[[-0.03252999857068062, 0.00824000034481287, -...","[1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2, 2, 2, 1, ...",-0.053161,0.448001


## Plot

In [142]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='ls1', y='ls2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='ls1', y='ls2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('LS dim 1')
    plt.ylabel('LS dim 2')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    # sns.scatterplot(data=df_sample, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='pc1', y='pc2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='pc1', y='pc2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.tight_layout()
    plt.show()

# Sample every latent distrubution and calculate loss

In [118]:
# Number of samples per point
n_samples = 100 #1000
test_data = 'validation'

# Seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Sample from each latent distribution
data_names = []
data_samples = []
data_sample_type = []
data_crystal_types = []
ground_truth_cell_parameters = []
ground_truth_cell_positions = []
ground_truth_cell_atoms = []
for data_index in range(len(df_rec)):
    latent_mean = df_rec['latent_space_mean'].iloc[data_index]
    latent_std = df_rec['latent_space_std'].iloc[data_index]
    latent_dist = torch.distributions.Normal(loc=torch.tensor(latent_mean).float(), scale=torch.tensor(latent_std).float())
    latent_samples = latent_dist.sample((n_samples,))
    
    data_samples.append(torch.tensor(latent_mean))
    data_samples.extend([latent_sample for latent_sample in latent_samples])
    
    data_names.append(f'{data_index}')
    data_names.extend([f'{data_index}_{i}' for i in range(n_samples)])
    
    data_sample_type.append('mean')
    data_sample_type.extend(['sample' for i in range(n_samples)])
    
    data_crystal_types.extend([df_rec['crystalType'].iloc[data_index] for i in range(n_samples+1)])
    
    # Ground truth
    ground_truth_cell_parameters.extend([df_rec['cell_parameters'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_positions.extend([df_rec['cell_positions'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_atoms.extend([df_rec['cell_atoms'].iloc[data_index]]*(n_samples+1))
    
data_samples = torch.stack(data_samples)
ground_truth_cell_parameters = torch.tensor(ground_truth_cell_parameters)
ground_truth_cell_positions = torch.tensor(ground_truth_cell_positions)
ground_truth_cell_atoms = torch.tensor(ground_truth_cell_atoms)

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

data_index = torch.tensor(np.arange(len(data_names)))

# Define dataset
sample_dataset = TensorDataset(data_samples, data_index, comp_onehot.repeat(len(data_names), 1), ground_truth_cell_parameters, ground_truth_cell_positions, ground_truth_cell_atoms)

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'sample_type': [],
    'crystalType': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
    'reconstruction_loss': [],
}
    

In [119]:
# Loss functions
loss_fn_cell_parameters = torch.nn.MSELoss()
# loss_fn_cell_positions = torch.nn.MSELoss()
# loss_fn_cell_atoms = torch.nn.CrossEntropyLoss()
# loss_fn_kld = torch.nn.KLDivLoss()
loss_fn_cell_positions = weighted_MSELoss()
loss_fn_cell_atoms = weighted_CrossEntropyLoss()
loss_fn_latent_mean = torch.nn.MSELoss()
loss_fn_latent_std = torch.nn.MSELoss()

In [120]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition, cell_parameters_true, cell_positions_true, cell_atoms_true = batch
    sample_point = sample_point.to(device)
    sample_indeces = sample_index.to(device)
    composition = composition.to(device)
    cell_parameters_true = cell_parameters_true.to(device)
    cell_positions_true = cell_positions_true.to(device)
    cell_atoms_true = cell_atoms_true.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    for sample_index in sample_indeces:
        sample_results['name'].append(data_names[sample_index])
        sample_results['sample_type'].append(data_sample_type[sample_index])
        sample_results['crystalType'].append(data_crystal_types[sample_index])
    
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Make loss weights
    cell_positions_weights = torch.where(cell_positions_true != -1, 1, 0).float().to(device)
    cell_atoms_weights = torch.where(cell_atoms_true != 0, 1, 0.1).float().to(device)

    # Simplify atom identities
    if setup_json['training']['simplified_atom_identities']:
        # Map atom number 0 to logit 0 (No atom)
        cell_atoms_true = torch.where(cell_atoms_true == 0, 0, cell_atoms_true)
        # Map atom numbers of ligands to logit 1 (Ligand) # [1, 6, 7, 8, 9, 15, 16, 17, 34, 35, 53]
        for ligand in setup_json['training']['ligands']:
            cell_atoms_true = torch.where(cell_atoms_true == ligand, 1, cell_atoms_true)
        # Map all other atom numbers to logit 2 (Metal)
        cell_atoms_true = torch.where(cell_atoms_true >= 2, 2, cell_atoms_true)
    
    # Calculate reconstruction loss for each sample
    for batch_index in range(this_batch_size):
        loss_cell_parameters = loss_fn_cell_parameters(cell_parameters[batch_index], cell_parameters_true[batch_index])
        loss_cell_positions = loss_fn_cell_positions(cell_positions[batch_index], cell_positions_true[batch_index], cell_positions_weights[batch_index])
        loss_cell_atoms = loss_fn_cell_atoms(cell_atoms[batch_index], cell_atoms_true[batch_index], cell_atoms_weights[batch_index])
        
        loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
        
        # Save loss
        sample_results['reconstruction_loss'].append(loss_reconstruction.item())
        
    # # Reconstruction loss
    # loss_cell_parameters = loss_fn_cell_parameters(cell_parameters, cell_parameters_true) 
    
    # # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true) # Unweighted
    # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true, cell_positions_weights) # Weighted
    
    # # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true) # Unweighted
    # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true, cell_atoms_weights) # Weighted
    
    # loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
    
    # # Save loss
    # sample_results['reconstruction_loss'].extend([loss_reconstruction.item()]*this_batch_size)


In [121]:
df_full_latent = pd.DataFrame(sample_results)
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    df_full_latent[['ls1', 'ls2']] = df_full_latent['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_full_latent[['pc1', 'pc2']] = pca.transform(np.array(df_full_latent['latent_point'].values.tolist())
)
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,pc1,pc2
0,0,mean,RheniumTrioxide,"[0.23553481698036194, 0.14010709524154663, 0.8...","[3.8670363426208496, 3.928786277770996, 3.8742...","[[0.0009899999713525176, 0.001560000004246831,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.075702,-0.708994,0.164626
1,0_0,sample,RheniumTrioxide,"[0.31157463788986206, 0.6433305144309998, 0.86...","[3.8712596893310547, 3.942988157272339, 3.9339...","[[-0.003819999983534217, -0.009909999556839466...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.304528,-0.754961,0.103098
2,0_1,sample,RheniumTrioxide,"[0.15244685113430023, 0.36965036392211914, 0.7...","[3.8668456077575684, 3.911404848098755, 3.8899...","[[-0.0018599999602884054, 0.012319999746978283...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.913904,-0.603571,0.199903
3,0_2,sample,RheniumTrioxide,"[0.23383529484272003, -0.40283292531967163, 0....","[3.8699734210968018, 3.9323372840881348, 3.911...","[[0.005690000019967556, 0.003930000122636557, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.207221,-0.708383,0.166478
4,0_3,sample,RheniumTrioxide,"[0.3005966544151306, 0.007311716675758362, 0.7...","[3.9192919731140137, 3.9981956481933594, 4.074...","[[0.00139999995008111, -0.019610000774264336, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.467968,-0.691076,0.079453


In [128]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)    
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.svg', dpi=300, transparent=True)

In [123]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_full_latent[contour_column].min()
max_loss = df_full_latent[contour_column].max()

# Plot
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['ls1'].min()-0.1, df_full_latent['ls1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['ls2'].min()-0.1, df_full_latent['ls2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['ls1'], df_full_latent['ls2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['pc1'].min()-0.1, df_full_latent['pc1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['pc2'].min()-0.1, df_full_latent['pc2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['pc1'], df_full_latent['pc2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_loss.pdf', dpi=300)	

In [124]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,pc1,pc2
0,0,mean,RheniumTrioxide,"[0.23553481698036194, 0.14010709524154663, 0.8...","[3.8670363426208496, 3.928786277770996, 3.8742...","[[0.0009899999713525176, 0.001560000004246831,...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.075702,-0.708994,0.164626
1,0_0,sample,RheniumTrioxide,"[0.31157463788986206, 0.6433305144309998, 0.86...","[3.8712596893310547, 3.942988157272339, 3.9339...","[[-0.003819999983534217, -0.009909999556839466...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.304528,-0.754961,0.103098
2,0_1,sample,RheniumTrioxide,"[0.15244685113430023, 0.36965036392211914, 0.7...","[3.8668456077575684, 3.911404848098755, 3.8899...","[[-0.0018599999602884054, 0.012319999746978283...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.913904,-0.603571,0.199903
3,0_2,sample,RheniumTrioxide,"[0.23383529484272003, -0.40283292531967163, 0....","[3.8699734210968018, 3.9323372840881348, 3.911...","[[0.005690000019967556, 0.003930000122636557, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.207221,-0.708383,0.166478
4,0_3,sample,RheniumTrioxide,"[0.3005966544151306, 0.007311716675758362, 0.7...","[3.9192919731140137, 3.9981956481933594, 4.074...","[[0.00139999995008111, -0.019610000774264336, ...","[2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, ...",0.467968,-0.691076,0.079453


In [125]:
# Plot all samples in 3d if latent space is 3D

# if len(df_rec['latent_space_mean'].iloc[0]) == 3:
    
#     df_crystal = df_full_latent.copy()
#     df_crystal[['ls1', 'ls2', 'ls3']] = df_crystal['latent_point'].apply(pd.Series)
    
#     fig = px.scatter_3d(df_crystal, x='ls1', y='ls2', z='ls3', color='crystalType', symbol='crystalType', hover_data=['name', 'ls1', 'ls2', 'ls3', 'reconstruction_loss', 'cell_parameters'], color_discrete_sequence=px.colors.qualitative.Dark24)
    
#     fig.update_layout(
#         scene = dict(
#             xaxis_title='LS dim 1',
#             yaxis_title='LS dim 2',
#             zaxis_title='LS dim 3',
#         ),
#         margin=dict(l=0, r=0, t=0, b=0),
#     )
    # fig.show()